# VeriFact: Colab quickstart
Runtime → Change runtime type → **T4 GPU**. Run the cells top to bottom.
Artifacts are saved to Google Drive so a disconnect doesn't lose your trained models.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/verifact'
!mkdir -p $DRIVE

## 1. Get the code
Either clone your GitHub repo, or upload `verifact.zip` to Drive and unzip it.

In [ ]:
# Option A: your GitHub repo
# !git clone https://github.com/<your-username>/verifact.git /content/verifact
# Option B: zip on Drive
!unzip -q -o $DRIVE/verifact.zip -d /content/
%cd /content/verifact
!pip install -q -r requirements.txt
!python -m pytest -q

## 2. Data

In [ ]:
!python scripts/prepare_data.py fever --inspect

In [ ]:
# If --inspect shows different column names, pass --claim-col / --evidence-col / --label-col
!python scripts/prepare_data.py fever
!git clone --depth 1 https://github.com/utahnlp/x-fact data/raw/x-fact

## 3. Baselines (CPU, a few minutes)

In [ ]:
!python scripts/train_tfidf.py --mode pair   --train data/processed/fever/train.jsonl --max-train 60000 --out artifacts/tfidf_pair.joblib
!python scripts/train_tfidf.py --mode single --train data/processed/fever/train.jsonl --max-train 60000 --out artifacts/tfidf_claim.joblib
!python scripts/evaluate.py --model artifacts/tfidf_pair.joblib  --data data/processed/fever/dev.jsonl
!python scripts/evaluate.py --model artifacts/tfidf_claim.joblib --data data/processed/fever/dev.jsonl

## 4. Verifier stage 1: English FEVER (GPU)
Start small (`data.max_train=20000`, 1 epoch) to check everything works, then do the full run.

In [ ]:
!python scripts/train.py --config configs/nli_fever.yaml --set data.max_train=20000 train.epochs=1 \
    output_dir=$DRIVE/artifacts/nli_fever
!ln -sfn $DRIVE/artifacts artifacts_drive

## 5. Hindi data + verifier stage 2

In [ ]:
!python scripts/translate_pairs.py --input data/processed/fever/train.jsonl --n 6000 \
    --out data/processed/fever_hi/translated.jsonl
!cp data/processed/fever_hi/translated.jsonl $DRIVE/

In [ ]:
!python scripts/prepare_data.py hindi --xfact-dir data/raw/x-fact --langs all \
    --translated data/processed/fever_hi/translated.jsonl \
    --mix-fever data/processed/fever/train.jsonl --mix-n 3000
!python scripts/train.py --config configs/nli_hindi.yaml \
    --set model.name=$DRIVE/artifacts/nli_fever/best output_dir=$DRIVE/artifacts/nli_hindi

## 6. Cross-dataset evaluation

In [ ]:
!python scripts/evaluate.py --model $DRIVE/artifacts/nli_hindi/best --data \
    data/processed/hindi/test.jsonl data/processed/hindi/xfact_ood.jsonl \
    data/processed/hindi/translated_test.jsonl data/processed/fever/dev.jsonl \
    --out $DRIVE/results/nli_hindi.json

## 7. Corpus, indexes and demo
Add fact-check documents from `scripts/fetch_factchecks.py` with `--docs` for a realistic knowledge base.

In [ ]:
!python scripts/prepare_data.py corpus --docs data/sample/corpus_docs.jsonl --from-pairs data/processed/hindi/train.jsonl
!python scripts/build_index.py --config configs/pipeline.yaml
!python scripts/verify.py "सरकार 1 दिसंबर से सभी 500 के नोट बंद कर रही है" \
    --set verifier.model_dir=$DRIVE/artifacts/nli_hindi/best

In [ ]:
# Streamlit through a Cloudflare tunnel (prints a public *.trycloudflare.com URL)
!sed -i "s#model_dir: artifacts/nli_hindi/best#model_dir: $DRIVE/artifacts/nli_hindi/best#" configs/pipeline.yaml
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
!nohup streamlit run app/streamlit_app.py --server.port 8501 > streamlit.log 2>&1 &
!sleep 5 && ./cloudflared tunnel --url http://localhost:8501